# Privacy-Preserving Anomaly Detection
## Federated Vending Machines

**Paper:** Privacy-Preserving Anomaly Detection for Payment Security in Vending Machines Using Federated Deep Autoencoders

**Author:** K. Rishi J - Jain University, Bangalore

---

**Run:** `Runtime -> Run all` - works immediately with synthetic data, no setup needed.

| Cell | Purpose |
|------|---------|
| 1 | Install packages |
| 2 | Write all source files |
| 3 | Upload real datasets (optional) |
| 4 | Run federated simulation |
| 5 | Full results dashboard (7 plots) |
| 6 | Per-node metrics table |
| 7 | DP privacy-accuracy tradeoff |
| 8 | Live transaction inference |
| 9 | Download results ZIP |


In [ ]:
# CELL 1 - Install packages
import subprocess, sys
pkgs = ['torch', 'flwr[simulation]', 'scikit-learn',
        'pandas', 'numpy', 'matplotlib', 'seaborn']
for p in pkgs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', p, '-q'])
print('All packages installed successfully')


In [ ]:
# CELL 2 - Write all source files
import os, sys
os.makedirs('data/raw', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('federated', exist_ok=True)
sys.path.insert(0, '.')

with open("config.py", 'w') as _f:
    _f.write("CONFIG = {\n    \"seq_len\":        10,\n    \"input_dim\":      28,\n    \"hidden_dim\":     64,\n    \"latent_dim\":     16,\n    \"local_epochs\":   3,\n    \"learning_rate\":  1e-3,\n    \"batch_size\":     64,\n    \"num_rounds\":     10,\n    \"num_clients\":    4,\n    \"fraction_fit\":   1.0,\n    \"dp_sigma\":       0.005,\n    \"threshold_pct\":  95,\n}\n\nNODE_NAMES = {\n    0: \"Node A \u2014 Office (Credit Card Fraud)\",\n    1: \"Node B \u2014 Campus (NSL-KDD)\",\n    2: \"Node C \u2014 Transport (UNSW-NB15)\",\n    3: \"Node D \u2014 Retail (Mixed)\",\n}\n")
with open("data/__init__.py", 'w') as _f:
    _f.write("")
with open("data/partition.py", 'w') as _f:
    _f.write("\"\"\"\nPartitions real or synthetic data into 4 non-IID federated nodes.\n\nPlace real datasets in data/raw/:\n  creditcard.csv             -> https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud\n  KDDTrain+.txt              -> https://www.unb.ca/cic/datasets/nsl.html\n  UNSW_NB15_training-set.csv -> https://research.unsw.edu.au/projects/unsw-nb15-dataset\n\nMissing files fall back to synthetic data automatically.\n\"\"\"\nimport os\nimport numpy as np\nimport pandas as pd\nfrom sklearn.preprocessing import StandardScaler\n\nRAW_DIR = os.path.join(os.path.dirname(os.path.abspath(__file__)), \"raw\")\n\n\ndef _synthetic(n=8000, feats=28, anom_rate=0.05, seed=0):\n    rng = np.random.default_rng(seed)\n    n_norm = int(n * (1 - anom_rate))\n    n_anom = n - n_norm\n    X = np.vstack([\n        rng.normal(0, 1,   (n_norm, feats)).astype(np.float32),\n        rng.normal(3, 1.5, (n_anom, feats)).astype(np.float32),\n    ])\n    y = np.array([0]*n_norm + [1]*n_anom)\n    idx = rng.permutation(n)\n    return X[idx], y[idx]\n\n\ndef _load_creditcard():\n    p = os.path.join(RAW_DIR, \"creditcard.csv\")\n    if not os.path.exists(p):\n        return None\n    df = pd.read_csv(p)\n    X  = df[[f\"V{i}\" for i in range(1,29)]].values.astype(np.float32)\n    y  = df[\"Class\"].values\n    return X, y\n\n\ndef _load_nslkdd():\n    p = os.path.join(RAW_DIR, \"KDDTrain+.txt\")\n    if not os.path.exists(p):\n        return None\n    cols = [f\"f{i}\" for i in range(41)] + [\"label\",\"difficulty\"]\n    df   = pd.read_csv(p, header=None, names=cols)\n    for c in [\"f1\",\"f2\",\"f3\"]:\n        df[c] = pd.Categorical(df[c]).codes\n    X = df[[f\"f{i}\" for i in range(28)]].fillna(0).values.astype(np.float32)\n    y = (df[\"label\"] != \"normal\").astype(int).values\n    return X, y\n\n\ndef _load_unswnb15():\n    p = os.path.join(RAW_DIR, \"UNSW_NB15_training-set.csv\")\n    if not os.path.exists(p):\n        return None\n    df   = pd.read_csv(p)\n    drop = [\"id\",\"attack_cat\",\"proto\",\"service\",\"state\"]\n    df   = df.drop(columns=[c for c in drop if c in df.columns])\n    lbl  = \"label\" if \"label\" in df.columns else df.columns[-1]\n    y    = df[lbl].values\n    cols = [c for c in df.columns if c != lbl][:28]\n    X    = df[cols].fillna(0).values.astype(np.float32)\n    return X, y\n\n\ndef _scale_split(X, y, split=0.8, seed=0):\n    rng  = np.random.default_rng(seed)\n    idx  = rng.permutation(len(X))\n    X, y = X[idx], y[idx]\n    n    = int(len(X)*split)\n    sc   = StandardScaler()\n    Xtr  = sc.fit_transform(X[:n])\n    Xte  = sc.transform(X[n:])\n    return Xtr.astype(np.float32), y[:n], Xte.astype(np.float32), y[n:]\n\n\ndef get_node_partitions():\n    cc   = _load_creditcard()\n    kdd  = _load_nslkdd()\n    unsw = _load_unswnb15()\n\n    nodes = {}\n\n    # Node 0 \u2014 Credit Card Fraud (Office)\n    if cc:\n        X, y = cc\n        Xtr,ytr,Xte,yte = _scale_split(X,y,seed=0)\n        nodes[0] = dict(X_train=Xtr,y_train=ytr,X_test=Xte,y_test=yte,\n                        name=\"Node A \u2014 Office\",source=\"Credit Card Fraud (Kaggle)\",\n                        using_synthetic=False)\n    else:\n        X,y = _synthetic(8000,28,0.002,seed=0)\n        Xtr,ytr,Xte,yte = _scale_split(X,y,seed=0)\n        nodes[0] = dict(X_train=Xtr,y_train=ytr,X_test=Xte,y_test=yte,\n                        name=\"Node A \u2014 Office\",source=\"Synthetic (Credit Card proxy)\",\n                        using_synthetic=True)\n\n    # Node 1 \u2014 NSL-KDD (Campus)\n    if kdd:\n        X,y = kdd\n        rng = np.random.default_rng(1)\n        idx = rng.permutation(min(20000,len(X)))\n        X,y = X[idx],y[idx]\n        Xtr,ytr,Xte,yte = _scale_split(X,y,seed=1)\n        nodes[1] = dict(X_train=Xtr,y_train=ytr,X_test=Xte,y_test=yte,\n                        name=\"Node B \u2014 Campus\",source=\"NSL-KDD (UNB)\",\n                        using_synthetic=False)\n    else:\n        X,y = _synthetic(8000,28,0.30,seed=1)\n        Xtr,ytr,Xte,yte = _scale_split(X,y,seed=1)\n        nodes[1] = dict(X_train=Xtr,y_train=ytr,X_test=Xte,y_test=yte,\n                        name=\"Node B \u2014 Campus\",source=\"Synthetic (NSL-KDD proxy)\",\n                        using_synthetic=True)\n\n    # Node 2 \u2014 UNSW-NB15 (Transport)\n    if unsw:\n        X,y = unsw\n        rng = np.random.default_rng(2)\n        idx = rng.permutation(min(20000,len(X)))\n        X,y = X[idx],y[idx]\n        Xtr,ytr,Xte,yte = _scale_split(X,y,seed=2)\n        nodes[2] = dict(X_train=Xtr,y_train=ytr,X_test=Xte,y_test=yte,\n                        name=\"Node C \u2014 Transport\",source=\"UNSW-NB15 (UNSW)\",\n                        using_synthetic=False)\n    else:\n        X,y = _synthetic(8000,28,0.20,seed=2)\n        Xtr,ytr,Xte,yte = _scale_split(X,y,seed=2)\n        nodes[2] = dict(X_train=Xtr,y_train=ytr,X_test=Xte,y_test=yte,\n                        name=\"Node C \u2014 Transport\",source=\"Synthetic (UNSW proxy)\",\n                        using_synthetic=True)\n\n    # Node 3 \u2014 Mixed (Retail)\n    parts = [d for d in [cc,kdd,unsw] if d is not None]\n    if len(parts) >= 2:\n        rng = np.random.default_rng(3)\n        Xs,ys = [],[]\n        for d in parts[:2]:\n            i = rng.permutation(min(4000,len(d[0])))\n            Xs.append(d[0][i]); ys.append(d[1][i])\n        X = np.vstack(Xs); y = np.concatenate(ys)\n        src = \"Mixed (CC + NSL-KDD)\"; syn = False\n    else:\n        X,y = _synthetic(8000,28,0.05,seed=3)\n        src = \"Synthetic (Mixed proxy)\"; syn = True\n    Xtr,ytr,Xte,yte = _scale_split(X,y,seed=3)\n    nodes[3] = dict(X_train=Xtr,y_train=ytr,X_test=Xte,y_test=yte,\n                    name=\"Node D \u2014 Retail\",source=src,using_synthetic=syn)\n\n    return nodes\n")
with open("data/dataset.py", 'w') as _f:
    _f.write("import numpy as np\nimport torch\nfrom torch.utils.data import Dataset, DataLoader\nfrom config import CONFIG\n\n\nclass SequenceDataset(Dataset):\n    def __init__(self, X, y, normal_only=False):\n        seq_len = CONFIG[\"seq_len\"]\n        seqs, lbls = [], []\n        for i in range(len(X) - seq_len + 1):\n            lbl = int(y[i + seq_len - 1])\n            if normal_only and lbl == 1:\n                continue\n            seqs.append(X[i:i+seq_len])\n            lbls.append(lbl)\n        self.seqs = np.array(seqs, dtype=np.float32)\n        self.lbls = np.array(lbls, dtype=np.int64)\n\n    def __len__(self):  return len(self.seqs)\n    def __getitem__(self, i):\n        return torch.tensor(self.seqs[i]), torch.tensor(self.lbls[i])\n\n\ndef make_loaders(node):\n    bs = CONFIG[\"batch_size\"]\n    tr = DataLoader(SequenceDataset(node[\"X_train\"],node[\"y_train\"],normal_only=True),\n                    batch_size=bs, shuffle=True,  drop_last=True)\n    ev = DataLoader(SequenceDataset(node[\"X_test\"], node[\"y_test\"],normal_only=False),\n                    batch_size=bs, shuffle=False, drop_last=False)\n    return tr, ev\n")
with open("models/__init__.py", 'w') as _f:
    _f.write("")
with open("models/autoencoder.py", 'w') as _f:
    _f.write("import torch\nimport torch.nn as nn\nfrom config import CONFIG\n\n\nclass LSTMAutoencoder(nn.Module):\n    def __init__(self):\n        super().__init__()\n        D = CONFIG[\"input_dim\"]\n        H = CONFIG[\"hidden_dim\"]\n        L = CONFIG[\"latent_dim\"]\n        S = CONFIG[\"seq_len\"]\n        self.seq_len = S\n        self.enc_lstm = nn.LSTM(D, H, num_layers=2, batch_first=True, dropout=0.1)\n        self.enc_fc   = nn.Linear(H, L)\n        self.dec_fc   = nn.Linear(L, H)\n        self.dec_lstm = nn.LSTM(H, D, num_layers=2, batch_first=True, dropout=0.1)\n\n    def forward(self, x):\n        _, (h, _) = self.enc_lstm(x)\n        z = self.enc_fc(h[-1])\n        d = self.dec_fc(z).unsqueeze(1).repeat(1, self.seq_len, 1)\n        out, _ = self.dec_lstm(d)\n        return out\n\n\ndef recon_error(x, x_hat):\n    return torch.mean((x - x_hat) ** 2, dim=(1, 2))\n")
with open("models/threshold.py", 'w') as _f:
    _f.write("import numpy as np\nfrom config import CONFIG\n\n\nclass AdaptiveThreshold:\n    def __init__(self):\n        self.threshold = None\n\n    def fit(self, normal_errors):\n        self.threshold = float(np.percentile(normal_errors, CONFIG[\"threshold_pct\"]))\n\n    def predict(self, errors):\n        return (np.array(errors) > self.threshold).astype(int)\n")
with open("federated/__init__.py", 'w') as _f:
    _f.write("")
with open("federated/client.py", 'w') as _f:
    _f.write("import time\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport flwr as fl\nfrom collections import OrderedDict\nfrom sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score\n\nfrom models.autoencoder import LSTMAutoencoder, recon_error\nfrom models.threshold   import AdaptiveThreshold\nfrom config import CONFIG\n\n\nclass VendingClient(fl.client.NumPyClient):\n    def __init__(self, node_id, train_loader, eval_loader):\n        self.node_id      = node_id\n        self.train_loader = train_loader\n        self.eval_loader  = eval_loader\n        self.model        = LSTMAutoencoder()\n        self.opt          = torch.optim.Adam(self.model.parameters(),\n                                             lr=CONFIG[\"learning_rate\"])\n        self.criterion    = nn.MSELoss()\n        self.threshold    = AdaptiveThreshold()\n\n    def get_parameters(self, config):\n        return [v.cpu().numpy() for v in self.model.state_dict().values()]\n\n    def set_parameters(self, params):\n        keys  = list(self.model.state_dict().keys())\n        state = OrderedDict({k: torch.tensor(v) for k,v in zip(keys,params)})\n        self.model.load_state_dict(state, strict=True)\n\n    def fit(self, params, config):\n        self.set_parameters(params)\n        self.model.train()\n        total_loss = 0; n = 0\n        for _ in range(CONFIG[\"local_epochs\"]):\n            for x, _ in self.train_loader:\n                self.opt.zero_grad()\n                loss = self.criterion(self.model(x), x)\n                loss.backward()\n                self.opt.step()\n                total_loss += loss.item(); n += 1\n        avg = total_loss / max(n,1)\n        return self.get_parameters({}), len(self.train_loader.dataset), {\"loss\": avg}\n\n    def evaluate(self, params, config):\n        self.set_parameters(params)\n        self.model.eval()\n        errors, labels, lats = [], [], []\n\n        with torch.no_grad():\n            # Fit threshold on normal samples\n            norm_errs = []\n            for x, lbl in self.eval_loader:\n                e = recon_error(x, self.model(x)).numpy()\n                norm_errs.extend(e[lbl.numpy()==0].tolist())\n            if norm_errs:\n                self.threshold.fit(np.array(norm_errs))\n            else:\n                self.threshold.threshold = 0.1\n\n            for x, lbl in self.eval_loader:\n                t0 = time.perf_counter()\n                e  = recon_error(x, self.model(x)).numpy()\n                lats.append((time.perf_counter()-t0)*1000/len(x))\n                errors.extend(e.tolist())\n                labels.extend(lbl.numpy().tolist())\n\n        preds = self.threshold.predict(np.array(errors))\n        labels = np.array(labels)\n\n        try:    auc = float(roc_auc_score(labels, errors))\n        except: auc = 0.0\n\n        metrics = {\n            \"node_id\":    self.node_id,\n            \"f1\":         round(float(f1_score(labels,preds,zero_division=0)),4),\n            \"precision\":  round(float(precision_score(labels,preds,zero_division=0)),4),\n            \"recall\":     round(float(recall_score(labels,preds,zero_division=0)),4),\n            \"auc_roc\":    round(auc,4),\n            \"latency_ms\": round(float(np.mean(lats)),3),\n            \"recon_error\":round(float(np.mean(errors)),6),\n            \"threshold\":  round(float(self.threshold.threshold),6),\n            \"normal_errors\":  [round(e,6) for e in norm_errs[:200]],\n            \"anomaly_errors\": [round(e,6) for e in\n                               np.array(errors)[labels==1][:200].tolist()],\n        }\n        return float(np.mean(errors)), len(self.eval_loader.dataset), metrics\n")
with open("federated/strategy.py", 'w') as _f:
    _f.write("import numpy as np\nimport flwr as fl\nfrom flwr.server.strategy import FedAvg\nfrom flwr.common import parameters_to_ndarrays, ndarrays_to_parameters\n\n\nclass FedAvgDP(FedAvg):\n    \"\"\"FedAvg + Gaussian differential privacy noise.\"\"\"\n\n    def __init__(self, dp_sigma=0.005, *args, **kwargs):\n        super().__init__(*args, **kwargs)\n        self.dp_sigma     = dp_sigma\n        self.round_log    = []   # [{round, avg_loss, avg_f1, node_results}]\n\n    def aggregate_fit(self, rnd, results, failures):\n        agg = super().aggregate_fit(rnd, results, failures)\n        if agg is None: return None\n        params, metrics = agg\n        # Inject DP noise\n        noisy = [a + np.random.normal(0, self.dp_sigma, a.shape)\n                 for a in parameters_to_ndarrays(params)]\n        losses = [r.metrics.get(\"loss\",0) for _,r in results]\n        self.round_log.append({\n            \"round\":    rnd,\n            \"avg_loss\": round(float(np.mean(losses)),6),\n        })\n        return ndarrays_to_parameters(noisy), metrics\n\n    def aggregate_evaluate(self, rnd, results, failures):\n        agg = super().aggregate_evaluate(rnd, results, failures)\n        if agg is None: return None\n        loss, metrics = agg\n        node_results = [r.metrics for _,r in results]\n        avg_f1 = round(float(np.mean([n.get(\"f1\",0) for n in node_results])),4)\n        # Attach to matching round log entry\n        for entry in reversed(self.round_log):\n            if entry[\"round\"] == rnd:\n                entry[\"avg_f1\"]       = avg_f1\n                entry[\"node_results\"] = node_results\n                break\n        return loss, metrics\n")
with open("federated/runner.py", 'w') as _f:
    _f.write("import sys, os\nsys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))\n\nimport flwr as fl\nfrom data.partition   import get_node_partitions\nfrom data.dataset     import make_loaders\nfrom federated.client   import VendingClient\nfrom federated.strategy import FedAvgDP\nfrom config import CONFIG\n\n\ndef run_simulation(dp_sigma=None, num_rounds=None, verbose=True):\n    dp_sigma   = dp_sigma   or CONFIG[\"dp_sigma\"]\n    num_rounds = num_rounds or CONFIG[\"num_rounds\"]\n\n    if verbose: print(\"Loading datasets...\")\n    nodes   = get_node_partitions()\n    loaders = {i: make_loaders(n) for i,n in nodes.items()}\n\n    # Print dataset summary\n    if verbose:\n        print(f\"\\n{'Node':<30} {'Source':<35} {'Train':>7} {'Test':>6} {'Anom%':>6}\")\n        print(\"-\"*85)\n        for i,n in nodes.items():\n            ar = round(n[\"y_test\"].mean()*100,2)\n            syn = \" [synthetic]\" if n[\"using_synthetic\"] else \"\"\n            print(f\"{n['name']:<30} {n['source']+syn:<35} \"\n                  f\"{len(n['X_train']):>7} {len(n['X_test']):>6} {ar:>5}%\")\n        print()\n\n    def client_fn(cid):\n        nid = int(cid)\n        tr, ev = loaders[nid]\n        return VendingClient(nid, tr, ev).to_client()\n\n    strategy = FedAvgDP(\n        dp_sigma=dp_sigma,\n        fraction_fit=1.0,\n        min_fit_clients=CONFIG[\"num_clients\"],\n        min_evaluate_clients=CONFIG[\"num_clients\"],\n        min_available_clients=CONFIG[\"num_clients\"],\n    )\n\n    if verbose:\n        print(f\"Starting simulation: {num_rounds} rounds | \"\n              f\"{CONFIG['num_clients']} nodes | DP \u03c3={dp_sigma}\\n\")\n\n    import logging\n    logging.getLogger(\"flwr\").setLevel(logging.WARNING)\n\n    fl.simulation.start_simulation(\n        client_fn=client_fn,\n        num_clients=CONFIG[\"num_clients\"],\n        config=fl.server.ServerConfig(num_rounds=num_rounds),\n        strategy=strategy,\n        client_resources={\"num_cpus\":1,\"num_gpus\":0.0},\n    )\n\n    return strategy.round_log, nodes\n")

print('All source files written')


In [ ]:
# CELL 3 - (Optional) Upload real datasets
# Skip this cell to use synthetic data - works perfectly.
# To use real data, upload any of:
#   creditcard.csv             -> kaggle.com/datasets/mlg-ulb/creditcardfraud
#   KDDTrain+.txt              -> unb.ca/cic/datasets/nsl.html
#   UNSW_NB15_training-set.csv -> research.unsw.edu.au/projects/unsw-nb15-dataset
import os
os.makedirs('data/raw', exist_ok=True)
try:
    from google.colab import files
    print('Upload dataset files (or Cancel to skip)')
    up = files.upload()
    for name, data in up.items():
        with open(f'data/raw/{name}', 'wb') as f:
            f.write(data)
        print(f'  Saved {name} ({len(data)//1024} KB)')
    if not up:
        print('  No files uploaded - using synthetic data')
except Exception as e:
    print(f'  Skipped ({e}) - using synthetic data')


In [ ]:
# CELL 4 - Run Federated Simulation
import sys
sys.path.insert(0, '.')

NUM_ROUNDS = 10    # number of federated rounds
DP_SIGMA   = 0.005 # differential privacy noise

from federated.runner import run_simulation
round_log, nodes = run_simulation(dp_sigma=DP_SIGMA, num_rounds=NUM_ROUNDS)
print(f'Simulation complete: {len(round_log)} rounds logged')


In [ ]:
# CELL 5 - Full Results Dashboard
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

BG     = '#0f1117'
BG2    = '#1a1d27'
BORDER = '#2e3248'
TEXT   = '#e2e4f0'
MUTED  = '#7c82a0'
COLORS = ['#4f8ef7', '#2dd4bf', '#a78bfa', '#fb7185']
TEAL   = '#2dd4bf'
AMBER  = '#f59e0b'
GREEN  = '#22c55e'
RED    = '#f87171'

plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': BG2,
    'axes.edgecolor': BORDER, 'axes.labelcolor': MUTED,
    'xtick.color': MUTED, 'ytick.color': MUTED,
    'text.color': TEXT, 'grid.color': BORDER,
    'grid.linestyle': '--', 'grid.alpha': 0.5,
})

rounds  = [r['round']       for r in round_log]
losses  = [r['avg_loss']    for r in round_log]
f1s     = [r.get('avg_f1', 0) for r in round_log]
nr      = round_log[-1].get('node_results', [])
NN = ['Node A\nOffice', 'Node B\nCampus',
      'Node C\nTransport', 'Node D\nRetail']

fig = plt.figure(figsize=(18, 14), facecolor=BG)
fig.suptitle('Federated Vending Machine - Anomaly Detection Results',
             color=TEXT, fontsize=16, fontweight='bold', y=0.98)
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

# Plot 1: F1 per round
ax = fig.add_subplot(gs[0, 0])
ax.plot(rounds, f1s, color=TEAL, lw=2, marker='o', ms=4)
ax.fill_between(rounds, f1s, alpha=0.15, color=TEAL)
ax.set_title('F1 Score per Round', color=TEXT, fontsize=11, pad=8)
ax.set_xlabel('Round'); ax.set_ylabel('Avg F1')
ax.set_ylim(0, 1); ax.grid(True)

# Plot 2: Loss per round
ax = fig.add_subplot(gs[0, 1])
ax.plot(rounds, losses, color=AMBER, lw=2, marker='s', ms=4)
ax.fill_between(rounds, losses, alpha=0.15, color=AMBER)
ax.set_title('Training Loss per Round', color=TEXT, fontsize=11, pad=8)
ax.set_xlabel('Round'); ax.set_ylabel('Loss'); ax.grid(True)

# Plot 3: Per-node F1 bar
ax = fig.add_subplot(gs[0, 2])
nf1 = [r.get('f1', 0) for r in nr]
bars = ax.bar(range(len(nf1)), nf1,
              color=COLORS[:len(nf1)], width=0.6, edgecolor=BORDER, lw=0.5)
for b, v in zip(bars, nf1):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.01,
            f'{v:.3f}', ha='center', va='bottom', fontsize=9, color=TEXT)
ax.set_title('Per-Node F1 (Last Round)', color=TEXT, fontsize=11, pad=8)
ax.set_xticks(range(len(NN[:len(nf1)])))
ax.set_xticklabels(NN[:len(nf1)], fontsize=8)
ax.set_ylim(0, 1.15); ax.grid(True, axis='y')

# Plot 4: Reconstruction error distribution
ax = fig.add_subplot(gs[1, 0:2])
if nr:
    n0  = nr[0]
    ne  = n0.get('normal_errors', [])
    ae  = n0.get('anomaly_errors', [])
    thr = n0.get('threshold')
    if ne: ax.hist(ne, bins=40, color=TEAL, alpha=0.7, label='Normal',  density=True)
    if ae: ax.hist(ae, bins=40, color=RED,  alpha=0.7, label='Anomaly', density=True)
    if thr:
        ax.axvline(thr, color=AMBER, lw=2, ls='--',
                   label='Threshold=' + str(round(thr, 4)))
    ax.set_title('Reconstruction Error Distribution - Node A',
                 color=TEXT, fontsize=11, pad=8)
    ax.set_xlabel('Reconstruction Error'); ax.set_ylabel('Density')
    ax.legend(facecolor=BG2, edgecolor=BORDER, labelcolor=TEXT, fontsize=9)
    ax.grid(True)

# Plot 5: Latency per node
ax = fig.add_subplot(gs[1, 2])
lats = [r.get('latency_ms', 0) for r in nr]
lcols = [GREEN if l < 50 else AMBER for l in lats]
ax.barh(range(len(lats)), lats, color=lcols, edgecolor=BORDER, lw=0.5)
ax.axvline(50, color=RED, ls='--', lw=1.5, label='50ms target')
for i, v in enumerate(lats):
    ax.text(v + 0.1, i, str(round(v, 2)) + 'ms',
            va='center', fontsize=9, color=TEXT)
ax.set_yticks(range(len(NN[:len(lats)])))
ax.set_yticklabels(NN[:len(lats)], fontsize=8)
ax.set_title('Inference Latency per Node', color=TEXT, fontsize=11, pad=8)
ax.set_xlabel('ms')
ax.legend(facecolor=BG2, edgecolor=BORDER, labelcolor=TEXT, fontsize=9)
ax.grid(True, axis='x')

# Plot 6: Precision / Recall / F1 grouped
ax = fig.add_subplot(gs[2, 0:2])
x     = np.arange(len(nr))
w     = 0.25
precs = [r.get('precision', 0) for r in nr]
recs  = [r.get('recall', 0)    for r in nr]
fvals = [r.get('f1', 0)        for r in nr]
ax.bar(x - w, precs, w, label='Precision', color='#4f8ef7',
       edgecolor=BORDER, lw=0.5)
ax.bar(x,     recs,  w, label='Recall',    color='#a78bfa',
       edgecolor=BORDER, lw=0.5)
ax.bar(x + w, fvals, w, label='F1',        color=TEAL,
       edgecolor=BORDER, lw=0.5)
ax.set_title('Precision / Recall / F1 per Node',
             color=TEXT, fontsize=11, pad=8)
ax.set_xticks(x)
ax.set_xticklabels(NN[:len(nr)], fontsize=8)
ax.set_ylim(0, 1.2); ax.grid(True, axis='y')
ax.legend(facecolor=BG2, edgecolor=BORDER, labelcolor=TEXT, fontsize=9)

# Plot 7: AUC-ROC
ax = fig.add_subplot(gs[2, 2])
aucs = [r.get('auc_roc', 0) for r in nr]
bars = ax.bar(range(len(aucs)), aucs,
              color=COLORS[:len(aucs)], width=0.6, edgecolor=BORDER, lw=0.5)
for b, v in zip(bars, aucs):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.01,
            str(round(v, 3)), ha='center', va='bottom', fontsize=9, color=TEXT)
ax.axhline(0.5, color=RED, ls='--', lw=1, label='Random baseline')
ax.set_title('AUC-ROC per Node', color=TEXT, fontsize=11, pad=8)
ax.set_xticks(range(len(NN[:len(aucs)])))
ax.set_xticklabels(NN[:len(aucs)], fontsize=8)
ax.set_ylim(0, 1.15); ax.grid(True, axis='y')
ax.legend(facecolor=BG2, edgecolor=BORDER, labelcolor=TEXT, fontsize=9)

plt.savefig('results_dashboard.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('Dashboard saved as results_dashboard.png')


In [ ]:
# CELL 6 - Per-Node Results Table
from IPython.display import HTML
nr   = round_log[-1].get('node_results', [])
NN   = ['Node A - Office', 'Node B - Campus',
        'Node C - Transport', 'Node D - Retail']
SR   = ['Credit Card Fraud', 'NSL-KDD', 'UNSW-NB15', 'Mixed']
CL   = ['#4f8ef7', '#2dd4bf', '#a78bfa', '#fb7185']
rows = ''
for i, n in enumerate(nr):
    f1  = n.get('f1', 0)
    fc  = '#22c55e' if f1 >= 0.8 else '#f59e0b' if f1 >= 0.6 else '#f87171'
    lat = n.get('latency_ms', 0)
    lc  = '#22c55e' if lat < 50 else '#f87171'
    syn = nodes[i].get('using_synthetic', False) if i in nodes else False
    src = SR[i] + (' (synthetic)' if syn else '')
    dot = '<span style="display:inline-block;width:9px;height:9px;border-radius:50%;'
    dot += 'background:' + CL[i] + ';margin-right:7px"></span>'
    rows += '<tr>'
    rows += '<td>' + dot + NN[i] + '</td>'
    rows += '<td style="color:#7c82a0;font-size:11px">' + src + '</td>'
    rows += '<td style="color:' + fc + ';font-weight:700">' + str(round(f1, 4)) + '</td>'
    rows += '<td>' + str(round(n.get('precision', 0), 4)) + '</td>'
    rows += '<td>' + str(round(n.get('recall', 0), 4)) + '</td>'
    rows += '<td>' + str(round(n.get('auc_roc', 0), 4)) + '</td>'
    rows += '<td style="color:' + lc + '">' + str(round(lat, 2)) + 'ms</td>'
    rows += '<td style="color:#7c82a0;font-size:11px">'
    rows += str(round(n.get('threshold', 0), 5)) + '</td></tr>'
avg_f1 = round_log[-1].get('avg_f1', 0)
html = '''
<style>
.rt{border-collapse:collapse;width:100%;font-family:monospace;font-size:13px}
.rt th{background:#1a1d27;color:#7c82a0;padding:10px 14px;border-bottom:1px solid #2e3248;
       font-size:11px;text-transform:uppercase;letter-spacing:0.5px}
.rt td{padding:10px 14px;border-bottom:1px solid #1a1d27;color:#e2e4f0}
.rt caption{caption-side:top;color:#2dd4bf;font-size:15px;font-weight:700;padding:10px 0}
</style>
<table class=rt>
<caption>Results | Avg F1: ''' + str(round(avg_f1,4)) + ''' | Rounds: ''' + str(len(round_log)) + ''' | DP sigma=''' + str(DP_SIGMA) + '''</caption>
<thead><tr><th>Node</th><th>Dataset</th><th>F1</th><th>Precision</th>
<th>Recall</th><th>AUC-ROC</th><th>Latency</th><th>Threshold</th></tr></thead>
<tbody>''' + rows + '''</tbody></table>'''
from IPython.display import display
display(HTML(html))


In [ ]:
# CELL 7 - DP Privacy-Accuracy Tradeoff
# Runs simulation with 5 different sigma values to show privacy-accuracy curve
# QUICK=True uses 3 rounds per sigma (fast), False uses 10 rounds (accurate)
import logging, numpy as np, matplotlib.pyplot as plt
logging.getLogger('flwr').setLevel(logging.ERROR)

QUICK  = True
sigmas = [0.0, 0.001, 0.005, 0.01, 0.05]
dp_f1s = []
rc     = 3 if QUICK else 10

print('Running DP tradeoff experiment...')
for sig in sigmas:
    lg, _ = run_simulation(dp_sigma=sig, num_rounds=rc, verbose=False)
    best  = max((r.get('avg_f1', 0) for r in lg), default=0)
    dp_f1s.append(best)
    print('  sigma=' + str(sig) + '  ->  F1=' + str(round(best, 4)))

BG='#0f1117'; BG2='#1a1d27'; BORDER='#2e3248'
TEXT='#e2e4f0'; MUTED='#7c82a0'
TEAL='#2dd4bf'; AMBER='#f59e0b'; RED='#f87171'

fig, ax = plt.subplots(figsize=(9, 4), facecolor=BG)
ax.set_facecolor(BG2)
ax.plot(sigmas, dp_f1s, color=TEAL, lw=2.5, marker='o',
        ms=9, markerfacecolor=AMBER, markeredgecolor=AMBER)
for s, f in zip(sigmas, dp_f1s):
    ax.annotate(str(round(f, 3)), (s, f),
                textcoords='offset points', xytext=(6, 7),
                color=TEXT, fontsize=10)
ax.axvline(DP_SIGMA, color=RED, ls='--', lw=1.5,
           label='Default sigma=' + str(DP_SIGMA))
ax.set_xlabel('DP Noise Sigma  (higher = more privacy)', color=MUTED)
ax.set_ylabel('Best F1 Score', color=MUTED)
ax.set_title('Privacy-Accuracy Tradeoff', color=TEXT, fontsize=12)
ax.legend(facecolor=BG2, edgecolor=BORDER, labelcolor=TEXT)
ax.grid(True)
for sp in ax.spines.values(): sp.set_edgecolor(BORDER)
plt.tight_layout()
plt.savefig('dp_tradeoff.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('Saved dp_tradeoff.png')


In [ ]:
# CELL 8 - Live Transaction Inference Demo
import torch, numpy as np
from models.autoencoder import LSTMAutoencoder, recon_error
from IPython.display import HTML, display

def infer(features, threshold=0.1):
    v = np.array(features, dtype=np.float32)
    v = np.pad(v, (0, max(0, 28 - len(v))))[:28]
    x = torch.tensor(np.tile(v, (10, 1))).unsqueeze(0)
    m = LSTMAutoencoder(); m.eval()
    with torch.no_grad():
        err = recon_error(x, m(x)).item()
    score = min(err / (threshold * 2), 1.0)
    return round(err, 6), round(score, 4), err > threshold

tests = [
    ('Normal transaction',
     [0.1,-0.2,0.5,0.3,-0.1,0.4,-0.3,0.2,0.1,-0.5,
      0.3,0.1,-0.2,0.4,0.2,-0.1,0.3,-0.4,0.1,0.2,
      -0.3,0.1,0.5,-0.2,0.3,-0.1,0.4,0.2]),
    ('Suspicious - extreme values',
     [5.2,-8.1,12.3,-6.4,9.1,-11.2,7.8,-9.3,
      13.1,-7.5,10.2,-8.8,6.9,-12.1,8.4,-10.3,
      11.7,-7.2,9.8,-8.6,12.4,-6.7,10.9,-9.1,
      7.3,-11.8,8.2,-10.5]),
    ('Replay attack - repeating spikes',
     [0,0,0,0,100,0,0,0,0,0,0,0,100,0,0,0,
      0,0,0,100,0,0,0,0,0,0,0,100]),
    ('Card skimming - tiny amounts',
     [0.01] * 28),
]

rows = ''
for name, feats in tests:
    err, score, ia = infer(feats)
    vc = '#f87171' if ia else '#22c55e'
    vt = 'ANOMALY' if ia else 'NORMAL'
    bw = int(score * 100)
    rows += '<tr>'
    rows += '<td>' + name + '</td>'
    rows += '<td style="font-family:monospace">' + str(err) + '</td>'
    rows += '<td><div style="width:120px;background:#1a1d27;border-radius:3px;height:12px">'
    rows += '<div style="width:' + str(bw) + '%;height:100%;background:' + vc
    rows += ';border-radius:3px"></div></div>'
    rows += '<span style="font-size:11px;color:' + vc + '">'
    rows += str(round(score*100, 1)) + '%</span></td>'
    rows += '<td style="color:' + vc + ';font-weight:700">' + vt + '</td></tr>'

display(HTML(
    '<style>.it{border-collapse:collapse;width:100%;font-family:monospace;font-size:13px}'
    '.it th{background:#1a1d27;color:#7c82a0;padding:10px 14px;'
    'border-bottom:1px solid #2e3248;font-size:11px;text-transform:uppercase}'
    '.it td{padding:10px 14px;border-bottom:1px solid #1a1d27;color:#e2e4f0}'
    '.it caption{caption-side:top;color:#2dd4bf;font-size:15px;'
    'font-weight:700;padding:10px 0}</style>'
    '<table class=it><caption>Live Inference Results</caption>'
    '<thead><tr><th>Transaction</th><th>Recon Error</th>'
    '<th>Anomaly Score</th><th>Verdict</th></tr></thead>'
    '<tbody>' + rows + '</tbody></table>'
))


In [ ]:
# CELL 9 - Download All Results
import json, zipfile, os
from google.colab import files

with open('round_log.json', 'w') as f:
    json.dump(round_log, f, indent=2, default=str)

with zipfile.ZipFile('results.zip', 'w') as z:
    for fn in ['results_dashboard.png', 'dp_tradeoff.png', 'round_log.json']:
        if os.path.exists(fn):
            z.write(fn)
            print('  added: ' + fn)

files.download('results.zip')
print('results.zip downloaded - use plots directly in your paper')
